# Transformers and Attention 

### Basic Form of Self-Attention

<image src="self_attn_basic.png" width="600">

In [1]:
import torch

# integer mapping of some sentence
sentence = torch.tensor(
    [0, # can
     7, # you     
     1, # help
     2, # me
     5, # to
     6, # translate
     4, # this
     3] # sentence
)

In [2]:
torch.manual_seed(123)
embed = torch.nn.Embedding(10, 16)
embedded_sentence = embed(sentence).detach()
print(embedded_sentence.shape)

torch.Size([8, 16])


In [3]:
# calculate the dot product between i-th and j-th word embeddings
omega = torch.empty(8,8)
for i, x_i in enumerate(embedded_sentence):
    for j, x_j in enumerate(embedded_sentence):
        omega[i, j] = x_i.dot(x_j)

In [4]:
# the O(n^2) for loop is super inefficient so just do matrix multiplication
omega_mat = embedded_sentence.matmul(embedded_sentence.T)

In [5]:
# verify that the two tensors are the same
torch.allclose(omega_mat, omega)

True

In [6]:
# calculate the attention weights 
import torch.nn.functional as F 
attn_weights = F.softmax(omega_mat, dim=1)
attn_weights.shape

torch.Size([8, 8])

*i-th* row of the attention weight matrix contains corresponding attention weights for all words in the sentence for the *i-th* word. Each attention weight indicates how relevant each word is to the *i-th* word.

In [7]:
# sun of the cols should sum to 1.0 since softmax normalizes the weights
attn_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [8]:
# compute the context vector
context_vectors = torch.matmul(attn_weights, embedded_sentence)
print(context_vectors.shape)

torch.Size([8, 16])


### Scaled Dot Product Attention

Three weight matrices (query, key, and value) are introduced as learnable parameters.

Below diagram shows how to compute the context vector for the 2nd input element.

<image src="dot_prod_attn.png" width="600">

In [11]:
# compute the context-aware embedding vector corresponding to the 2nd input element
torch.manual_seed(123)
d = embedded_sentence.shape[1]
print(f"d = {d}")
U_query = torch.rand(d,d)
U_key = torch.rand(d,d)
U_val = torch.rand(d,d)

d = 16


In [21]:
# using the query projection matrix, compute the query sequence
x_2 = embedded_sentence[1]
query_2 = U_query.matmul(x_2)
key_2 = U_key.matmul(x_2)
val_2 = U_val.matmul(x_2)

print(f"key_2 shape : {key_2.shape}")

# in order to compute context-aware embedding vector, need ALL key and val sequences from 
# all other input elements
keys = U_key.matmul(embedded_sentence.T).T
vals = U_val.matmul(embedded_sentence.T).T
print(f"Keys shape: {keys.shape}")

key_2 shape : torch.Size([16])
Keys shape: torch.Size([8, 16])


In [23]:
# confirm that keys and vals are correct
print(torch.allclose(key_2, keys[1]))
print(torch.allclose(val_2, vals[1]))

True
True


In [61]:
# compute omegas
omega_23 = query_2.dot(keys[2])
print(omega_23)

# scale up to all keys
omega_2s = query_2.matmul(keys.T)
print(omega_2s)

tensor(14.3667)
tensor([-25.1623,   9.3602,  14.3667,  32.1482,  53.8976,  46.6626,  -1.2131,
        -32.9392])


In [26]:
# normalize the weights omega
attn_weights_2 = F.softmax(omega_2s/d**0.5, dim=0) # divide by sqrt(m), m = d_k = d to ensure 
                                                   # Euclidean length of weight vectors appx. the same
print(attn_weights_2)

tensor([2.2317e-09, 1.2499e-05, 4.3696e-05, 3.7242e-03, 8.5596e-01, 1.4026e-01,
        8.8897e-07, 3.1935e-10])


In [27]:
# output z is weighted average of value sequences
context_vec_2 = attn_weights_2.matmul(vals)
print(context_vec_2)

tensor([-1.2226, -3.4387, -4.3928, -5.2125, -1.1249, -3.3041, -1.4316, -3.2765,
        -2.5114, -2.6105, -1.5793, -2.8433, -2.4142, -0.3998, -1.9917, -3.3499])


### Transformer Architecture

<image src="trans_archi.png" width=400>

There are two main blocks: *encoder* and a *decoder*. The encoder receives the original sequential input and encodes the embeddings as a multi-head self-attention module. Decoder takes in the processed input and outputs the resulting sequence.

### Multi-head Attention

In normal scaled dot product attention, there were 3 matrices: query, key and value. For multi-head attention, this set of 3 matrices can be thought of as *one head*.

In [34]:
torch.manual_seed(123)
d = embedded_sentence.shape[1] # embedding dim
one_U_query = torch.rand(d,d)

In [35]:
# multiple heads can simply be added by adding an additional dimension
h = 8
multihead_U_query = torch.rand(h,d,d)
multihead_U_key = torch.rand(h,d,d)
multihead_U_val = torch.rand(h,d,d)

In [36]:
# calculate query sequence for 2nd input in the 2nd head
print(multihead_U_query.shape)
print(x_2.shape)
multihead_query_2 = multihead_U_query.matmul(x_2)
print(multihead_query_2.shape)

torch.Size([8, 16, 16])
torch.Size([16])
torch.Size([8, 16])


`multihead_query_2` has eight matrix rows, where each row corresponds to the j-th head.

In [44]:
# compute key and val sequences for all heads
multihead_key_2 = multihead_U_key.matmul(x_2)
multihead_val_2 = multihead_U_val.matmul(x_2)
print("Second head key sequence for 2nd input element: \n", multihead_key_2[1])

Second head key sequence for 2nd input element: 
 tensor([-3.4011, -1.9002, -0.3384, -5.0635, -2.4668, -1.1029, -0.7261, -1.3852,
        -0.4445, -2.4184,  0.4935,  0.1313, -0.3554, -0.9890,  0.3959, -1.7146])


In [40]:
# repeat key and val computations for all input sequence elements
stacked_inputs = embedded_sentence.T.repeat(h,1,1)
print(stacked_inputs.shape)

# use batch matrix multiplication
multihead_keys = multihead_U_key.bmm(stacked_inputs)
print(multihead_keys.shape)

torch.Size([8, 16, 8])
torch.Size([8, 16, 8])


In [41]:
# swap dims of num_words and embedding_dim for more intuitive representation
multihead_keys = multihead_keys.permute(0, 2, 1)
print(multihead_keys.shape)

torch.Size([8, 8, 16])


In [45]:
# access second key value in second attention head
print("Second head key sequence for 2nd input element: \n", multihead_keys[1,1])

Second head key sequence for 2nd input element: 
 tensor([-3.4011, -1.9002, -0.3384, -5.0635, -2.4668, -1.1029, -0.7261, -1.3852,
        -0.4445, -2.4184,  0.4935,  0.1313, -0.3554, -0.9890,  0.3959, -1.7146])


In [48]:
multihead_values = multihead_U_val.bmm(stacked_inputs).permute(0,2,1)
print(multihead_values.shape)

torch.Size([8, 8, 16])


In [69]:
# book skips computing the context vector z_2 for multi-head...will do myself

# compute omegas (should be of dim 8 heads x 8 weights)
print(f"Multihead Query shape: {multihead_query_2.shape}") # 8 heads each with 16 values
print(f"Multihead Key shape: {multihead_keys.shape}")

multihead_omegas = torch.einsum("bij,ij->bi", multihead_keys, multihead_query_2)
print(F"Multihead Omega shape: {multihead_omegas.shape}")

# calculate attention weights
multihead_attn_weights = F.softmax(multihead_omegas/d**0.5, dim=0)
print(f"Multihead alpha shape: {multihead_attn_weights.shape}")

# calculate context vectors
print(f"Multihead Val shape: {multihead_values.shape}")
multihead_context_vec_2 = torch.einsum("bij,ii->bj", multihead_values, multihead_attn_weights)
print(f"Multihead Context Vector 2 shape: {multihead_context_vec_2.shape}")


Multihead Query shape: torch.Size([8, 16])
Multihead Key shape: torch.Size([8, 8, 16])
Multihead Omega shape: torch.Size([8, 8])
Multihead alpha shape: torch.Size([8, 8])
Multihead Val shape: torch.Size([8, 8, 16])
Multihead Context Vector 2 shape: torch.Size([8, 16])
